<a href="https://colab.research.google.com/github/Suresh-kg/oasis/blob/main/deep_fake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# (a) If you uploaded a ZIP directly into this Colab session
import os, zipfile, glob

zip_candidates = glob.glob('/content/*.zip')
print("Zip files found in /content:", zip_candidates)

if zip_candidates:
    zip_path = zip_candidates[0]
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/data')
    print(f"Extracted {zip_path} to /content/data")
else:
    print("No zip found — if data's already unzipped, skip this cell.")

Zip files found in /content: []
No zip found — if data's already unzipped, skip this cell.


In [ ]:
# (b) If your data is in Google Drive instead
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def find_dataset_root(search_dirs):
    for base in search_dirs:
        if not os.path.exists(base):
            continue
        for dirpath, dirnames, _ in os.walk(base):
            lower_dirs = [d.lower() for d in dirnames]
            has_train = "train" in lower_dirs
            has_val = ("valid" in lower_dirs) or ("val" in lower_dirs)
            has_test = "test" in lower_dirs
            if has_train and has_val and has_test:
                train_dir = os.path.join(dirpath, dirnames[lower_dirs.index("train")])
                sub = [d.lower() for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))]
                if "real" in sub and "fake" in sub:
                    return dirpath
    return None

search_dirs = ["/content/sample_data"]
DATA_ROOT = find_dataset_root(search_dirs)

if DATA_ROOT is None:
    print('Could not auto-detect. Set it manually, e.g.:')
    print('DATA_ROOT = "/content/sample_data"')
    DATA_ROOT = "/content/sample_data"  # fixed: was pointing at a .ipynb file, not a directory
else:
    print("Found dataset at:", DATA_ROOT)

Could not auto-detect. Set it manually, e.g.:
DATA_ROOT = "/content/sample_data"


In [ ]:
for split in ["train", "valid", "val", "test"]:
    p = os.path.join(DATA_ROOT, split)
    if os.path.exists(p):
        counts = {cls: len(os.listdir(os.path.join(p, cls))) for cls in os.listdir(p)
                  if os.path.isdir(os.path.join(p, cls))}
        print(f"{split}: {counts}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_curve, auc, precision_score, recall_score, f1_score)
import seaborn as sns

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 64
SEED = 42
VALID_DIRNAME = "valid" if os.path.exists(os.path.join(DATA_ROOT, "valid")) else "val"

train_datagen = ImageDataGenerator(
    rescale=1./255, rotation_range=20, width_shift_range=0.1,
    height_shift_range=0.1, zoom_range=0.1, horizontal_flip=True
)
eval_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    os.path.join(DATA_ROOT, "train"), target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode="binary", classes=["real", "fake"], seed=SEED)

val_gen = eval_datagen.flow_from_directory(
    os.path.join(DATA_ROOT, VALID_DIRNAME), target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode="binary", classes=["real", "fake"], shuffle=False)

test_gen = eval_datagen.flow_from_directory(
    os.path.join(DATA_ROOT, "test"), target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode="binary", classes=["real", "fake"], shuffle=False)

Found 0 images belonging to 2 classes.
Found 0 images belonging to 2 classes.
Found 0 images belonging to 2 classes.


In [ ]:
model = Sequential([
    Conv2D(32, (3,3), activation="relu", padding="same", input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    BatchNormalization(), MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation="relu", padding="same"),
    BatchNormalization(), MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation="relu", padding="same"),
    BatchNormalization(), MaxPooling2D(2,2),

    Conv2D(256, (3,3), activation="relu", padding="same"),
    BatchNormalization(), MaxPooling2D(2,2),

    Flatten(),
    Dense(256, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])
model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 16, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 16384)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     4,194,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,585,153 (17.49 MB)

 Trainable params: 4,584,193 (17.49 MB)

 Non-trainable params: 960 (3.75 KB)

In [ ]:
model.compile(optimizer=Adam(learning_rate=1e-4), loss="binary_crossentropy", metrics=["accuracy"])

callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
    ModelCheckpoint("best_deepfake_cnn.h5", monitor="val_loss", save_best_only=True)
]

In [ ]:
# NEW: this cell was missing entirely — without it the model is never trained
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=25,
    callbacks=callbacks
)

ValueError: The PyDataset has length 0

In [ ]:
# Optional but useful: evaluate on the held-out test set
test_loss, test_acc = model.evaluate(test_gen)
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f}")

y_true = test_gen.classes
y_pred_probs = model.predict(test_gen).ravel()
y_pred = (y_pred_probs >= 0.5).astype(int)

print(classification_report(y_true, y_pred, target_names=["real", "fake"]))

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["real", "fake"], yticklabels=["real", "fake"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix")
plt.show()

In [ ]:
model.save("deepfake_detector_cnn_v2.h5")
from google.colab import files
files.download("deepfake_detector_cnn_v2.h5")

In [ ]:
# NEW: inference on your uploaded image
from tensorflow.keras.preprocessing import image

img_path = "/content/young-bearded-man-with-striped-shirt.jpg"
img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

pred = model.predict(img_array)[0][0]
label = "fake" if pred >= 0.5 else "real"
print(f"Prediction score: {pred:.4f} -> {label}")

plt.imshow(img)
plt.title(f"Prediction: {label} ({pred:.4f})")
plt.axis("off")
plt.show()